# 02. Pose Extraction


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pose_extraction import extract_pose_sequence, batch_extract_and_cache, get_default_device
from src.preprocessing import load_cached_features

PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

print(f"Pose extraction device: {get_default_device()}")

Pose extraction device: 0


## 1. Single Video Test

In [2]:
manifest = pd.read_csv(ROOT / "data" / "metadata.csv")
squats = manifest[manifest["exercise"].str.contains("squat", case=False, na=False)]
sample_video = ROOT / squats.iloc[0]["video_path"]

print(f"Sample video: {sample_video.name}")
features_16 = extract_pose_sequence(sample_video, n_frames=16)
print(f"16-frame feature shape: {features_16.shape} (frames, features)")
print(f"Mean keypoint confidence: {features_16[:, 2::3].mean():.3f}")

features_32 = extract_pose_sequence(sample_video, n_frames=32)
print(f"32-frame feature shape: {features_32.shape} (frames, features)")

Sample video: subject_026_squat_good_diagonal.mp4


16-frame feature shape: (16, 51) (frames, features)
Mean keypoint confidence: 0.762


32-frame feature shape: (32, 51) (frames, features)


## 2. Check Cached Features

In [3]:
cache_16 = PROC / "squat_pose_16f_v1.npz"
if cache_16.exists():
    d16 = load_cached_features(cache_16)
    print(f"16-frame cache: {d16['X'].shape} | Clips: {len(d16['y'])} | Subjects: {len(np.unique(d16['groups']))}")

cache_32 = PROC / "squat_pose_32f_v1.npz"
if cache_32.exists():
    d32 = load_cached_features(cache_32)
    print(f"32-frame cache: {d32['X'].shape} | Clips: {len(d32['y'])} | Subjects: {len(np.unique(d32['groups']))}")
else:
    print(f"32-frame cache will be generated at {cache_32}")

16-frame cache: (150, 16, 51) | Clips: 150 | Subjects: 25
32-frame cache: (150, 32, 51) | Clips: 150 | Subjects: 25
